# SoftMoE-Retrieval — Train RIÊNG từng bộ DML
Hybrid CNN+ViT · Multi-scale TokenLearner · Soft MoE · RouteRank.

Mỗi bộ (CUB / Cars / In-Shop) = **một model riêng**, giao thức zero-shot chuẩn.

> Đổi bộ data: chỉ sửa **một biến `DATASET`** ở Mục 3, mọi cell phía dưới tự chạy theo.

## 1. Cài đặt

In [ ]:
# !pip install -r requirements.txt
# !pip install gdown          # cần cho việc tải data từ Google Drive
import torch; print('CUDA:', torch.cuda.is_available())

## 2. Lấy code (tự clone từ GitHub nếu chưa có)
Chạy được cả khi mở từ repo đã clone (local) lẫn môi trường trống (Colab).

In [ ]:
# ── Lấy code: tự clone repo nếu chưa có (Colab/máy mới); bỏ qua nếu đã có sẵn ──
import os, sys

REPO_URL = 'https://github.com/trong5nhan6/retrieval.git'
REPO_DIR = 'retrieval'      # tên thư mục sau khi clone

if os.path.exists('config.py'):                       # đang ở ngay trong repo
    PROJECT_ROOT = os.path.abspath('.')
elif os.path.exists(os.path.join('..', 'config.py')): # đang ở notebooks/ (clone local)
    PROJECT_ROOT = os.path.abspath('..')
else:                                                 # môi trường trống -> clone
    if not os.path.isdir(REPO_DIR):
        os.system(f'git clone {REPO_URL} {REPO_DIR}')
    PROJECT_ROOT = os.path.abspath(REPO_DIR)

os.chdir(PROJECT_ROOT)                # cwd = gốc project (để train.py chạy thẳng)
sys.path.insert(0, PROJECT_ROOT)
print('PROJECT_ROOT =', PROJECT_ROOT)

## 3. Chọn bộ data + tải từ Drive
**Chỉ cần sửa biến `DATASET`.** Bộ nào có link Drive sẽ tự tải `.zip` về `datasets/` và giải nén.

In [ ]:
from config import HCFG

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CHỌN BỘ DATA — chỉ sửa duy nhất dòng này: 'cub' | 'cars' | 'inshop'       ║
DATASET = 'cub'
# ╚══════════════════════════════════════════════════════════════════════════╝

# ── Link Google Drive cho từng bộ (.zip). Điền link rồi chọn DATASET tương ứng ─
DATASET_URLS = {
    'cub':    'https://drive.google.com/file/d/1bwP3KNX1Kaj-sPanx32KrCl4_NVM5m2P/view?usp=drive_link',
    'cars':   '',   # TODO: dán link Drive cho Cars196.zip
    'inshop': '',   # TODO: dán link Drive cho In-shop.zip
}

DATASETS_DIR = os.path.join(PROJECT_ROOT, 'datasets')
os.makedirs(DATASETS_DIR, exist_ok=True)

import re, zipfile

def _drive_id(url):
    m = re.search(r'/d/([A-Za-z0-9_-]+)', url) or re.search(r'[?&]id=([A-Za-z0-9_-]+)', url)
    return m.group(1) if m else None

def fetch_dataset(name):
    """Tải <name>.zip từ Drive vào datasets/ rồi giải nén (bỏ qua nếu chưa có link)."""
    url = DATASET_URLS.get(name, '')
    if not url:
        print(f'[{name}] chưa có link Drive — bỏ qua (hãy điền DATASET_URLS).')
        return
    import gdown
    zip_path = os.path.join(DATASETS_DIR, f'{name}.zip')
    if not os.path.exists(zip_path):
        print(f'[{name}] tải từ Drive...')
        gdown.download(id=_drive_id(url), output=zip_path, quiet=False)
    print(f'[{name}] giải nén -> {DATASETS_DIR}')
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(DATASETS_DIR)
    print(f'[{name}] xong.')

# Tải đúng bộ đang chọn
fetch_dataset(DATASET)

# ── Trỏ HCFG tới datasets/ (đường dẫn tuyệt đối) ─────────────────────────────
HCFG.data_roots = {
    'cub':    os.path.join(DATASETS_DIR, 'CUB_200_2011'),
    'cars':   os.path.join(DATASETS_DIR, 'Cars196'),
    'inshop': os.path.join(DATASETS_DIR, 'In-shop Clothes Retrieval Benchmark'),
}
HCFG.epochs = 60
print('DATASET =', DATASET, '| root =', HCFG.data_roots[DATASET])

## 4. Smoke-test shape (tải DINOv2 + ConvNeXt)

In [ ]:
from models.hybrid_encoder import HybridEncoder
from models.hyms_route import HyMSRoute
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
enc = HybridEncoder(HCFG.vit_name, HCFG.cnn_name, dev); enc.freeze_all()
m = HyMSRoute(enc, HCFG).to(dev)
z, rho, C = m(torch.randn(2, 3, 224, 224).to(dev))
print('z', z.shape, 'rho', rho.shape, 'C', C.shape)
print('head params:', sum(p.numel() for p in m.head_parameters() if p.requires_grad))

## 5. Train bộ đang chọn

In [ ]:
!python train.py --dataset {DATASET} --seed 42

## 6. Eval lại từ checkpoint (base vs RouteRank)

In [ ]:
from data.dml_dataset import get_dml_loaders
from eval.routerank import evaluate_self, evaluate_query_gallery

L = get_dml_loaders(DATASET, HCFG)
ck = torch.load(f'results/checkpoints/best_hyms_{DATASET}_seed42.pt', map_location=dev)
m.load_state_dict(ck['model'])
rk = HCFG.recall_k_for(DATASET)
if DATASET == 'inshop':                       # query/gallery rời nhau
    r = evaluate_query_gallery(m, L['query'], L['gallery'], dev, HCFG, recall_k=rk)
else:                                          # CUB/Cars: self-retrieval trên test
    r = evaluate_self(m, L['test'], dev, HCFG, recall_k=rk)
print('BASE     :', r['base']); print('RouteRank:', r['routerank'])

## Ablation (qua config)
`HCFG.cnn_stages=[1,2,3]` bỏ s1 · `HCFG.lambda_route=0` bỏ L_route · `HCFG.rr_beta=0` tắt RouteRank.